In [2]:
import pandas as pd

# Define paths
file_data = r"D:\UPEC PhD Project\UPEC_ML_Data\Frontiers_Word_Templates\Revision 2.0\Analysis\Datasets\Merged_ARGs_VFs_MEGs.csv"
file_mlst = r"D:\UPEC PhD Project\UPEC_ML_Data\Frontiers_Word_Templates\Revision 2.0\Analysis\Datasets\MLST.csv"
output_path = r"D:\UPEC PhD Project\UPEC_ML_Data\Frontiers_Word_Templates\Revision 2.0\Analysis\Datasets\Merged_ARGs_VFs_MGEs_MLST.csv"

# 1. Load data
df_data = pd.read_csv(file_data)
df_mlst = pd.read_csv(file_mlst)

# Ensure 'Isolate ID' is string to avoid merging errors
df_data['Isolate ID'] = df_data['Isolate ID'].astype(str).str.strip()
df_mlst['Isolate ID'] = df_mlst['Isolate ID'].astype(str).str.strip()

# 2. Merge
# We use 'left' join to keep all features from your cleaned dataset
# This adds the 'MLST' column to the dataframe
final_df = pd.merge(df_data, df_mlst[['Isolate ID', 'MLST']], on='Isolate ID', how='left')

# 3. Final Column Ordering
# Structure: [Isolate ID] + [All other columns except ID and MLST] + [MLST]
cols_to_keep = [c for c in final_df.columns if c not in ['Isolate ID', 'MLST']]
new_order = ['Isolate ID'] + cols_to_keep + ['MLST']

final_df = final_df[new_order]

# 4. Save
final_df.to_csv(output_path, index=False)

# 5. Verification
print("Merge and reordering complete!")
print(f"Final shape: {final_df.shape}")
print(f"Columns: {final_df.columns[0]} (left) ... {final_df.columns[-1]} (right)")

Merge and reordering complete!
Final shape: (1184, 2369)
Columns: Isolate ID (left) ... MLST (right)


In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

# ==========================================
# 1. FILE PATHS & SETUP
# ==========================================
input_file = r"D:\UPEC PhD Project\UPEC_ML_Data\Frontiers_Word_Templates\Revision 2.0\Analysis\Datasets\Merged_ARGs_VFs_MGEs_MLST.csv"
output_dir = r"D:\UPEC PhD Project\UPEC_ML_Data\Frontiers_Word_Templates\Revision 2.0\Analysis\Datasets\CoOccurrence_Results"

os.makedirs(output_dir, exist_ok=True)

print("Loading dataset...")
df = pd.read_csv(input_file, index_col=0)  # Column A (Isolate ID) becomes the index
print(f"Dataset successfully loaded. Shape: {df.shape}")

# ==========================================
# 2. EXCEL COLUMN TO PYTHON INDEX CONVERTER
# ==========================================
def excel_col_to_index(col_str):
    """
    Converts Excel column string coordinates (e.g., 'B', 'GS', 'CMB') into a 
    0-based Python column index.
    """
    col_str = col_str.upper().strip()
    num = 0
    for char in col_str:
        num = num * 26 + (ord(char) - ord('A') + 1)
    return num - 2

# ==========================================
# 3. RANGE-BASED ATTRIBUTE PARTITIONING
# ==========================================
arg_start, arg_end = excel_col_to_index('B'), excel_col_to_index('GS')
vf_start, vf_end = excel_col_to_index('GT'), excel_col_to_index('RB')
plas_start, plas_end = excel_col_to_index('RC'), excel_col_to_index('TJ')
int_start, int_end = excel_col_to_index('TK'), excel_col_to_index('TO')
is_start, is_end = excel_col_to_index('TP'), excel_col_to_index('UJ')
phage_start, phage_end = excel_col_to_index('UK'), excel_col_to_index('CMB')
mlst_idx = excel_col_to_index('CMC')

arg_cols = list(df.columns[arg_start : arg_end + 1])
vf_cols = list(df.columns[vf_start : vf_end + 1])
plasmid_cols = list(df.columns[plas_start : plas_end + 1])
integron_cols = list(df.columns[int_start : int_end + 1])
is_cols = list(df.columns[is_start : is_end + 1])
phage_cols = list(df.columns[phage_start : phage_end + 1])
mlst_col = df.columns[mlst_idx]

mge_cols = plasmid_cols + integron_cols + is_cols + phage_cols
target_cols = arg_cols + vf_cols

df.rename(columns={mlst_col: 'MLST'}, inplace=True)

# ==========================================
# 4. GLOBAL CO-OCCURRENCE ANALYSIS
# ==========================================
print("\nRunning Global Co-occurrence Analysis...")
global_results = []
valid_mges = [col for col in mge_cols if df[col].std() > 0]
valid_targets = [col for col in target_cols if df[col].std() > 0]

for mge in valid_mges:
    if mge in plasmid_cols: mge_type = "Plasmid"
    elif mge in integron_cols: mge_type = "Integron"
    elif mge in is_cols: mge_type = "IS Element"
    else: mge_type = "Phage"

    for target in valid_targets:
        r_val, p_val = pearsonr(df[mge], df[target])
        target_type = "ARG" if target in arg_cols else "VF"
        global_results.append({'MGE_Gene': mge, 'MGE_Type': mge_type, 'Target_Gene': target, 'Target_Type': target_type, 'Pearson_r': r_val, 'p_value': p_val})

global_df = pd.DataFrame(global_results)
if not global_df.empty:
    reject, pvals_corrected, _, _ = multipletests(global_df['p_value'], alpha=0.05, method='fdr_bh')
    global_df['p_adj_FDR'] = pvals_corrected
    global_df['Significant'] = reject
    global_sig = global_df[global_df['Significant'] & (global_df['Pearson_r'] > 0.3)].sort_values(by='Pearson_r', ascending=False)
    global_sig.to_csv(os.path.join(output_dir, "Global_MGE_Associations_Sig.csv"), index=False)

# ==========================================
# 5. LINEAGE-SPECIFIC CO-OCCURRENCE ANALYSIS (TOP 17 STs, EXCLUDING UNKNOWN)
# ==========================================
print("\nRunning Lineage-Specific Co-occurrence Analysis...")
# Filter out 'Unknown' and pick top 17 STs
st_counts = df[df['MLST'] != 'Unknown']['MLST'].value_counts()
predominant_sts = st_counts.nlargest(17).index.tolist()
print(f"Targeting top 17 predominant lineages: {predominant_sts}")

lineage_results = []
for st in predominant_sts:
    st_sub = df[df['MLST'] == st]
    st_mges = [col for col in mge_cols if st_sub[col].std() > 0]
    st_targets = [col for col in target_cols if st_sub[col].std() > 0]
    
    for mge in st_mges:
        mge_type = "Plasmid" if mge in plasmid_cols else "Integron" if mge in integron_cols else "IS Element" if mge in is_cols else "Phage"
        for target in st_targets:
            r_val, p_val = pearsonr(st_sub[mge], st_sub[target])
            lineage_results.append({'MLST': st, 'MGE_Gene': mge, 'MGE_Type': mge_type, 'Target_Gene': target, 'Pearson_r': r_val, 'p_value': p_val})

lineage_df = pd.DataFrame(lineage_results)
if not lineage_df.empty:
    corrected_dfs = []
    for st, group in lineage_df.groupby('MLST'):
        group = group.copy()
        reject, pvals_corrected, _, _ = multipletests(group['p_value'], alpha=0.05, method='fdr_bh')
        group['p_adj_FDR'] = pvals_corrected
        group['Significant'] = reject
        corrected_dfs.append(group)
    lineage_df_corrected = pd.concat(corrected_dfs)
    lineage_sig = lineage_df_corrected[lineage_df_corrected['Significant'] & (lineage_df_corrected['Pearson_r'] > 0.4)].sort_values(by=['MLST', 'Pearson_r'], ascending=[True, False])
    lineage_sig.to_csv(os.path.join(output_dir, "Lineage_Specific_MGE_Associations_Sig.csv"), index=False)

# ==========================================
# 6. VISUALIZATION (HEATMAP)
# ==========================================
if not global_sig.empty:
    top_pairs = global_sig.head(15)
    heatmap_data = []
    for idx, row in top_pairs.iterrows():
        mge, target = row['MGE_Gene'], row['Target_Gene']
        row_dict = {'Pair': f"{mge} ↔ {target}"}
        for st in predominant_sts:
            st_sub = df[df['MLST'] == st]
            if st_sub[mge].std() > 0 and st_sub[target].std() > 0:
                r_val, _ = pearsonr(st_sub[mge], st_sub[target])
                row_dict[f"ST{st}"] = r_val
            else:
                row_dict[f"ST{st}"] = np.nan
        heatmap_data.append(row_dict)
        
    heatmap_df = pd.DataFrame(heatmap_data).set_index('Pair')
    plt.figure(figsize=(15, 12))
    sns.heatmap(heatmap_df, cmap='coolwarm', vmin=-1, vmax=1, annot=True, fmt=".2f")
    plt.title("Top Co-occurrence Patterns across Top 17 STs")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "Lineage_MGE_CoOccurrence_Heatmap.png"), dpi=300)
    plt.close()

Loading dataset...
Dataset successfully loaded. Shape: (1184, 2368)

Running Global Co-occurrence Analysis...

Running Lineage-Specific Co-occurrence Analysis...
Targeting top 17 predominant lineages: ['ST131', 'ST1193', 'ST69', 'ST73', 'ST38', 'ST10', 'ST95', 'ST648', 'ST12', 'ST410', 'ST127', 'ST405', 'ST167', 'ST44', 'ST617', 'ST101', 'ST58']
